# RotaLivre | CitizenTech 2026.2

Equipe: 
Igor Davi Avelar Rosa Cesário — RM568163
Rafael Menezes Viana — RM576753
Patrick Castro Quintana — RM575906 · Turma: 2ESPS

Prova de conceito de rotas com custo de acessibilidade. Não é um sistema de navegação em produção.

## 1. Cenário e fonte dos dados
Pessoas com mobilidade reduzida podem precisar de desvios por barreiras físicas. O cenário tem origem A, destino D, dois caminhos alternativos e uma ligação bloqueada. E é isolado.

**Fonte:** conjunto sintético autoral, definido em TRECHOS e PONTOS para esta atividade, sem coleta de pessoas ou consulta a serviços externos. Distâncias e gravidades foram inventadas para tornar a demonstração reproduzível. Não há correspondência geográfica real.

**Referências do enunciado:** GS CitizenTech - 2ES Ago - FIAP 2026.pdf, páginas 1 e 2; GS - 2o Semestre de 2026.pdf, contexto geral CitizenTech fornecido com a atividade. Não são fontes dos valores simulados.

## 2. Modelagem
Grafo não direcionado em lista de adjacência: pontos são vértices e trechos são arestas. Cada trecho guarda metros, gravidade ordinal de 0 a 3 e bloqueio. A simetria é uma simplificação; um produto real deve representar direção, inclinação e perfis individuais.

Custo = metros + penalidade × gravidade. A penalidade é um parâmetro didático, expresso em unidades de custo por nível; custo não é tempo nem distância física. Bloqueios são excluídos mesmo com penalidade zero. Ausência de barreira no exemplo não certifica acessibilidade real.

In [1]:
from heapq import heappush, heappop
from math import inf

# Dados sintéticos: não representam ruas ou condições reais de São Paulo.
# Cada tupla: origem, destino, distância (m), gravidade (0..3), bloqueado.
TRECHOS = [
    ('A', 'B', 100, 3, False), ('B', 'D', 100, 0, False),
    ('A', 'C', 150, 0, False), ('C', 'D', 150, 0, False),
    ('A', 'D', 80, 0, True),
]
PONTOS = {'A': 'Origem', 'B': 'Calçada com barreira',
          'C': 'Desvio sem barreiras conhecidas', 'D': 'Destino',
          'E': 'Ponto isolado'}

def criar_grafo(trechos, pontos):
    grafo = {p: [] for p in pontos}
    for a, b, metros, gravidade, bloqueado in trechos:
        if a not in grafo or b not in grafo:
            raise ValueError('Ponto inexistente')
        if metros <= 0 or gravidade not in (0, 1, 2, 3):
            raise ValueError('Distância ou gravidade inválida')
        grafo[a].append((b, metros, gravidade, bloqueado))
        grafo[b].append((a, metros, gravidade, bloqueado))
    return grafo

def calcular_rota(grafo, origem, destino, penalidade=100):
    if origem not in grafo or destino not in grafo:
        raise ValueError('Origem ou destino inexistente')
    if penalidade < 0:
        raise ValueError('A penalidade deve ser não negativa')
    custos = {v: inf for v in grafo}
    anterior = {}
    custos[origem] = 0
    fila = [(0, origem)]
    while fila:
        custo, atual = heappop(fila)
        if custo != custos[atual]:
            continue
        if atual == destino:
            caminho = [destino]
            while caminho[-1] != origem:
                caminho.append(anterior[caminho[-1]])
            return {'caminho': caminho[::-1], 'custo': custo}
        for vizinho, metros, gravidade, bloqueado in grafo[atual]:
            if bloqueado:  # BLOQUEIO: barreira intransponível é excluída.
                continue
            peso = metros + penalidade * gravidade  # PESO: custo generalizado.
            novo = custo + peso
            if novo < custos[vizinho]:  # RELAXAMENTO: ideia central de Dijkstra.
                custos[vizinho] = novo
                anterior[vizinho] = atual
                heappush(fila, (novo, vizinho))
    return None

grafo = criar_grafo(TRECHOS, PONTOS)


## 3. Entradas e saídas
Entrada A → D, penalidade 0: caminho A-B-D, 200 m e custo 200. Com penalidade 100: A-C-D, 300 m e custo 300; A-B-D teria custo 500. O trecho direto de 80 m está bloqueado e nunca é usado. Não existe rota até E.

In [2]:
curta = calcular_rota(grafo, 'A', 'D', penalidade=0)
acessivel = calcular_rota(grafo, 'A', 'D', penalidade=100)
print('Menor distância entre trechos transitáveis:', curta)
print('Menor custo de acessibilidade:', acessivel)
print('Sem conexão:', calcular_rota(grafo, 'A', 'E'))


Menor distância entre trechos transitáveis: {'caminho': ['A', 'B', 'D'], 'custo': 200}
Menor custo de acessibilidade: {'caminho': ['A', 'C', 'D'], 'custo': 300}
Sem conexão: None


## 4. Testes
As verificações abaixo cobrem mudança de preferência, desconexão, origem igual ao destino, bloqueio obrigatório e entradas inválidas.

In [3]:
assert curta == {'caminho': ['A', 'B', 'D'], 'custo': 200}
assert acessivel == {'caminho': ['A', 'C', 'D'], 'custo': 300}
assert calcular_rota(grafo, 'A', 'E') is None
assert calcular_rota(grafo, 'A', 'A') == {'caminho': ['A'], 'custo': 0}
assert calcular_rota(grafo, 'A', 'D', 1000)['caminho'] == ['A', 'C', 'D']
assert calcular_rota(criar_grafo([('A','D',80,0,True)], ['A','D']), 'A','D') is None
for o, dest, p in [('X','D',100), ('A','D',-1)]:
    try:
        calcular_rota(grafo, o, dest, p)
    except ValueError:
        pass
    else:
        raise AssertionError('Entrada inválida aceita')
try:
    criar_grafo([('A','D',-1,0,False)], ['A','D'])
except ValueError:
    pass
else:
    raise AssertionError('Distância inválida aceita')
print('9 verificações concluídas com sucesso.')


9 verificações concluídas com sucesso.


## 5. Algoritmo, correção e eficiência
Dijkstra usa fila de prioridade mínima com heapq. O comentário RELAXAMENTO identifica a atualização de uma melhor estimativa; PESO e BLOQUEIO identificam as regras do cenário. Como todos os custos são não negativos, ao retirar uma entrada atualizada de menor custo, sua distância é definitiva. Se houvesse um caminho melhor, seu primeiro vértice ainda não finalizado teria sido alcançado a partir de um predecessor com custo menor, contradizendo a escolha da fila. Isso permite encerrar no destino.

Para V vértices e E arestas, a lista ocupa O(V+E). A implementação permite entradas repetidas no heap: tempo O((V+E) log(V+E)) e memória O(V+E). Em grafos simples conectados, a forma usual do tempo é O((V+E) log V). A reconstrução custa O(V).

Embora a disciplina se chame Dynamic Programming, o enunciado permite algoritmos de rota. Dijkstra é um algoritmo guloso, não programação dinâmica. Foi escolhido pela adequação a pesos não negativos.

## 6. Evolução e impacto
IA de visão computacional, foto/geolocalização, mapa colaborativo e integração com status de elevadores são propostas futuras, não implementadas. Uma classificação automática deve sugerir categoria e confiança e passar por confirmação; não garante que uma via esteja transitável. Dados vencidos e trechos desconhecidos precisam de tratamento explícito.

Métricas futuras: barreiras confirmadas/relatadas, idade dos relatos, taxa de rotas concluídas e tempo/esforço percebido em teste consentido. Nenhum impacto real foi medido. Conexão proposta: ODS 11, 10 e 9. Evitar publicar rostos, placas ou localização pessoal desnecessária.